# CallWhisper-8k: LoRA Adapter Reload Eval + Report Tables

This notebook reloads the committed Whisper-small LoRA adapter, evaluates it against the frozen GramVaani manifests, and exports report-ready tables for Excel.

It does **not** train. It is for reproducibility and reporting after the Kaggle pilot.


## What You Need In Drive

Put only the data files in Google Drive. The code, adapter weights, manifests, and runner come from GitHub.

Expected Drive layout:

```text
MyDrive/call-whisper/
  GV_Dev_5h/
    Audio/*.mp3
    text
    mp3.scp
  results/                    # notebook writes report outputs here
```

Optional, only if you also want clean-control eval later:

```text
MyDrive/call-whisper/clean_control/fleurs_hi_50/
  fleurs_hi_clean_50.csv
  audio/*.wav
```

You do **not** need to upload the LoRA model separately because it is now committed in the repo under:

```text
models/whisper-small-lora-gramvaani-pilot-seed0/
```


In [ ]:
# Runtime check
import sys, subprocess

print('Python:', sys.version)
subprocess.run(['nvidia-smi'], check=False)


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Clone or update the GitHub repo
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/anshulLuhsna/CallWhisper-8k.git'
REPO_DIR = Path('/content/CallWhisper-8k')
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/call-whisper')

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
os.environ['PYTHONPATH'] = str(REPO_DIR / 'src')

print('Repo:', REPO_DIR)
print('Drive project dir:', DRIVE_PROJECT_DIR)
print('Commit:')
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)


In [ ]:
# Install only reload/eval/report dependencies.
# This avoids training-notebook dependency churn.
# Colab may preinstall old torchao; PEFT 0.19 rejects torchao<0.16 even though
# this notebook does not need torchao for ordinary LoRA adapter loading.
%pip uninstall -y -q torchao
%pip install -q --no-deps -e .
%pip install -q 'jiwer>=3.0.4' 'librosa>=0.10' 'peft>=0.11' 'transformers>=4.41' 'accelerate>=0.30' 'soundfile>=0.12' 'tqdm>=4.66' 'pandas>=2.0' 'openpyxl>=3.1'


In [ ]:
# Link Drive audio into the cloned repo so the fixed manifests resolve.
import os
import shutil
from pathlib import Path

GV_DEV_DRIVE = DRIVE_PROJECT_DIR / 'GV_Dev_5h'
GV_DEV_REPO = REPO_DIR / 'datasets' / 'GV_Dev_5h'

if not GV_DEV_DRIVE.exists():
    raise FileNotFoundError(f'Missing Drive data folder: {GV_DEV_DRIVE}')

GV_DEV_REPO.parent.mkdir(parents=True, exist_ok=True)
if GV_DEV_REPO.exists() or GV_DEV_REPO.is_symlink():
    if GV_DEV_REPO.is_symlink() or GV_DEV_REPO.is_file():
        GV_DEV_REPO.unlink()
    else:
        shutil.rmtree(GV_DEV_REPO)
GV_DEV_REPO.symlink_to(GV_DEV_DRIVE, target_is_directory=True)

print('Linked:', GV_DEV_REPO, '->', GV_DEV_DRIVE)
print('Audio files visible:', len(list((GV_DEV_REPO / 'Audio').glob('*.mp3'))))


In [ ]:
# Configuration
from pathlib import Path

RUN_NAME = 'whisper-small-lora-gramvaani-pilot-seed0_colab_reload'
OUTPUT_DIR = Path('results/lora_reload_eval_colab')
DRIVE_RESULTS_DIR = DRIVE_PROJECT_DIR / 'results' / 'lora_reload_eval_colab'

MANIFESTS = [
    Path('datasets/manifests/gramvaani_dev_50.csv'),
    Path('datasets/manifests/gramvaani_dev_50_8khz.csv'),
    Path('datasets/manifests/gramvaani_dev_50_highrate.csv'),
]

# Optional FLEURS clean-control manifest if you already created it in Drive.
FLEURS_MANIFEST = DRIVE_PROJECT_DIR / 'clean_control/fleurs_hi_50/fleurs_hi_clean_50.csv'
if FLEURS_MANIFEST.exists():
    MANIFESTS.append(FLEURS_MANIFEST)
    print('Including optional FLEURS manifest:', FLEURS_MANIFEST)
else:
    print('FLEURS manifest not found, running GramVaani only.')

ADAPTER_DIR = Path('models/whisper-small-lora-gramvaani-pilot-seed0/final_adapter')
PROCESSOR_DIR = Path('models/whisper-small-lora-gramvaani-pilot-seed0/processor')

print('Manifests:')
for path in MANIFESTS:
    print(' -', path)
print('Adapter:', ADAPTER_DIR)
print('Processor:', PROCESSOR_DIR)


In [ ]:
# Quick manifest/audio validation before running models.
import csv

def resolve_manifest_audio(path_value, manifest_path):
    p = Path(path_value)
    if p.is_absolute():
        return p
    repo_candidate = REPO_DIR / p
    if repo_candidate.exists():
        return repo_candidate
    manifest_candidate = Path(manifest_path).parent / p
    return manifest_candidate

for manifest in MANIFESTS:
    rows = list(csv.DictReader(open(manifest, encoding='utf-8')))
    missing = []
    for row in rows:
        p = resolve_manifest_audio(row['audio_path'], manifest)
        if not p.exists():
            missing.append(str(p))
    print(manifest, 'rows=', len(rows), 'missing_audio=', len(missing))
    if missing[:3]:
        print(' first missing:', missing[:3])
    if missing:
        raise FileNotFoundError(f'{manifest} has missing audio files')


In [ ]:
# Preflight: imports, adapter files, and one audio decode check.
# Run this before the long eval cell so failures are obvious early.
import csv
import importlib.util
import json
import librosa
import subprocess
import torch
import transformers
import peft

print('Repo commit:')
subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], check=True)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('transformers:', transformers.__version__)
print('peft:', peft.__version__)
print('torchao installed:', importlib.util.find_spec('torchao') is not None)
if importlib.util.find_spec('torchao') is not None:
    raise RuntimeError('torchao is still installed. Run the install cell again so old Colab torchao is removed.')

required_adapter_files = [
    ADAPTER_DIR / 'adapter_config.json',
    ADAPTER_DIR / 'adapter_model.safetensors',
    PROCESSOR_DIR / 'preprocessor_config.json',
    PROCESSOR_DIR / 'tokenizer_config.json',
]
for path in required_adapter_files:
    print('exists', path, path.exists())
    if not path.exists():
        raise FileNotFoundError(path)

first_manifest = MANIFESTS[0]
with open(first_manifest, encoding='utf-8') as f:
    first_row = next(csv.DictReader(f))
first_audio = resolve_manifest_audio(first_row['audio_path'], first_manifest)
print('Trying one audio load:', first_audio)
array, sr = librosa.load(str(first_audio), sr=16000, mono=True, duration=3.0)
print('Loaded sample:', {'samples': len(array), 'sr': sr, 'seconds_loaded': round(len(array) / sr, 2)})
print('Preflight OK. Start eval cell next.')


In [ ]:
# Run same-pipeline base HF Whisper-small vs LoRA Whisper-small eval.
# Streams subprocess output live and prints a heartbeat if the model is quiet.
import queue
import subprocess
import sys
import threading
import time

cmd = [
    sys.executable, '-u', '-m', 'callwhisper.eval.lora_runner',
    '--base-model', 'openai/whisper-small',
    '--adapter-dir', str(ADAPTER_DIR),
    '--processor-dir', str(PROCESSOR_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--run-name', RUN_NAME,
    '--repo-root', str(REPO_DIR),
    '--language-mode', 'manifest',
    '--seed', '0',
]
for manifest in MANIFESTS:
    cmd.extend(['--manifest', str(manifest)])

print('RUN:', ' '.join(cmd), flush=True)
start_time = time.time()
line_queue = queue.Queue()

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env={**os.environ, 'PYTHONPATH': str(REPO_DIR / 'src')},
)

def reader():
    for line in proc.stdout:
        line_queue.put(line)

thread = threading.Thread(target=reader, daemon=True)
thread.start()
last_output = time.time()

while True:
    try:
        line = line_queue.get(timeout=5)
        print(line, end='', flush=True)
        last_output = time.time()
    except queue.Empty:
        pass

    return_code = proc.poll()
    now = time.time()
    if return_code is not None:
        while not line_queue.empty():
            print(line_queue.get(), end='', flush=True)
        break
    if now - last_output >= 30:
        elapsed_min = (now - start_time) / 60
        print(f'[heartbeat] eval still running: {elapsed_min:.1f} min elapsed', flush=True)
        last_output = now

if return_code != 0:
    raise subprocess.CalledProcessError(return_code, cmd)
print('Eval finished successfully.')


In [ ]:
# Build report tables for notebook display and Excel.
import json
import pandas as pd
from pathlib import Path

summary_path = OUTPUT_DIR / f'{RUN_NAME}_eval_summary.json'
comparison_path = OUTPUT_DIR / f'{RUN_NAME}_base_vs_lora_comparison.md'

summary_rows = json.loads(summary_path.read_text(encoding='utf-8'))
summary_df = pd.DataFrame(summary_rows)

keep_summary_cols = [
    'model', 'slice', 'condition', 'language_mode', 'num_beams', 'files',
    'macro_wer', 'macro_cer', 'corpus_wer', 'corpus_cer', 'file', 'adapter'
]
summary_table = summary_df[[col for col in keep_summary_cols if col in summary_df.columns]].copy()
for col in ['macro_wer', 'macro_cer', 'corpus_wer', 'corpus_cer']:
    if col in summary_table:
        summary_table[col] = summary_table[col].round(4)

base = summary_df[summary_df['model'] == 'hf_whisper_small_base'].copy()
lora = summary_df[summary_df['model'] == 'hf_whisper_small_lora'].copy()
comparison_df = base.merge(
    lora,
    on=['slice', 'condition', 'language_mode', 'num_beams', 'files'],
    suffixes=('_base', '_lora'),
)
comparison_table = comparison_df[[
    'slice', 'condition', 'num_beams', 'files',
    'macro_wer_base', 'macro_wer_lora',
    'macro_cer_base', 'macro_cer_lora',
    'corpus_wer_base', 'corpus_wer_lora',
    'corpus_cer_base', 'corpus_cer_lora',
]].copy()
comparison_table['delta_macro_wer'] = comparison_table['macro_wer_lora'] - comparison_table['macro_wer_base']
comparison_table['delta_macro_cer'] = comparison_table['macro_cer_lora'] - comparison_table['macro_cer_base']
comparison_table['delta_corpus_wer'] = comparison_table['corpus_wer_lora'] - comparison_table['corpus_wer_base']
comparison_table['delta_corpus_cer'] = comparison_table['corpus_cer_lora'] - comparison_table['corpus_cer_base']
for col in comparison_table.select_dtypes(include='number').columns:
    if col not in {'num_beams', 'files'}:
        comparison_table[col] = comparison_table[col].round(4)
comparison_table = comparison_table.sort_values(['slice', 'num_beams'])

samples = []
for json_file in sorted(OUTPUT_DIR.glob(f'{RUN_NAME}_hf_whisper_small_*_beam*.json')):
    payload = json.loads(json_file.read_text(encoding='utf-8'))
    for sample in payload['samples']:
        sample = dict(sample)
        sample['source_json'] = str(json_file)
        samples.append(sample)
samples_df = pd.DataFrame(samples)
if not samples_df.empty:
    for col in ['wer', 'cer']:
        samples_df[col] = samples_df[col].round(4)

print('Summary table')
display(summary_table)
print('Base vs LoRA comparison')
display(comparison_table)


In [ ]:
# Save report tables to Drive as Excel + CSV.
import shutil

DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

excel_path = DRIVE_RESULTS_DIR / f'{RUN_NAME}_report_tables.xlsx'
summary_csv = DRIVE_RESULTS_DIR / f'{RUN_NAME}_summary.csv'
comparison_csv = DRIVE_RESULTS_DIR / f'{RUN_NAME}_base_vs_lora_comparison.csv'
samples_csv = DRIVE_RESULTS_DIR / f'{RUN_NAME}_per_sample_predictions.csv'

summary_table.to_csv(summary_csv, index=False)
comparison_table.to_csv(comparison_csv, index=False)
samples_df.to_csv(samples_csv, index=False)

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    summary_table.to_excel(writer, sheet_name='summary', index=False)
    comparison_table.to_excel(writer, sheet_name='base_vs_lora', index=False)
    samples_df.to_excel(writer, sheet_name='per_sample', index=False)

# Copy raw JSON/Markdown artifacts too, so the report has backup evidence.
for path in OUTPUT_DIR.glob('*'):
    if path.is_file():
        shutil.copy2(path, DRIVE_RESULTS_DIR / path.name)

print('Saved Excel:', excel_path)
print('Saved CSV:', summary_csv)
print('Saved CSV:', comparison_csv)
print('Saved CSV:', samples_csv)
print('Drive folder:', DRIVE_RESULTS_DIR)


In [ ]:
# Optional direct browser download from Colab.
# Drive save above is the main path; this just gives you a clickable download too.
from google.colab import files

files.download(str(excel_path))


## How To Read The Tables In The Report

Use the `base_vs_lora` sheet for the main result. Negative deltas mean the LoRA adapter reduced error on that slice.

Suggested wording:

> On the fixed GramVaani slice, the Whisper-small LoRA adapter changed macro WER from X to Y under the same Hugging Face decoding pipeline.

Do not write that the adapter is globally better than Whisper or better than all Hindi ASR systems. This notebook only proves a same-pipeline base-vs-adapter result on the fixed slices you evaluated.
